<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp?1" width="100px"></a>
</td>
</tr>
</table>

# 使用 OpenAI API 评估指令回复

- 本 notebook 使用 OpenAI 的 GPT-4 API，基于 JSON 格式的数据集评估指令微调 LLM 的回复；该数据集包含模型生成的回复，例如：



```python
{
    "instruction": "What is the atomic number of helium?",
    "input": "",
    "output": "The atomic number of helium is 2.",               # <-- The target given in the test set
    "model 1 response": "\nThe atomic number of helium is 2.0.", # <-- Response by an LLM
    "model 2 response": "\nThe atomic number of helium is 3."    # <-- Response by a 2nd LLM
},
```

In [ ]:
# pip install -r requirements-extra.txt

In [ ]:
from importlib.metadata import version

pkgs = ["openai",  # OpenAI API
        "tqdm",    # 进度条
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

## 测试 OpenAI API

- 首先，让我们测试 OpenAI API 是否已正确配置
- 若尚无账户，需在 https://platform.openai.com/ 注册
- 注意，还需向账户充值，GPT-4 API 并非免费（见 https://platform.openai.com/settings/organization/billing/overview）
- 按本 notebook 中的代码运行实验并生成约 200 次评估，截至撰写时大约花费 $0.26（26 美分）

- 首先，我们需要提供 OpenAI API 密钥，可在 https://platform.openai.com/api-keys 获取
- 请勿与任何人共享此密钥
- 将此密钥（`"sk-..."`）添加到本文件夹中的 `config.json` 文件

In [ ]:
import json
from openai import OpenAI

# 从 JSON 文件加载 API 密钥。
# 请确保将 "sk-..." 替换为您从 https://platform.openai.com/api-keys 获取的实际 API 密钥
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

- 首先用简单示例测试 API，确保其按预期工作：

In [ ]:
def run_chatgpt(prompt, client, model="gpt-4-turbo"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        seed=123,
    )
    return response.choices[0].message.content


prompt = "Respond with 'hello world' if you got this message."
run_chatgpt(prompt, client)

## 加载 JSON 条目

- 此处假设我们将测试集与模型回复保存为 JSON 文件，可按如下方式加载：

In [ ]:
json_file = "eval-example-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("条目数量:", len(json_data))

- 该文件结构如下，其中包含测试集中的给定回复（`'output'`）以及两个不同模型的回复（`'model 1 response'` 和 `'model 2 response'`）：

In [ ]:
json_data[0]

- 下面是一个小型工具函数，用于格式化输入，便于后续可视化：

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- 现在，让我们尝试使用 OpenAI API 比较模型回复（我们仅评估前 5 条回复以便直观对比）：

In [ ]:
for entry in json_data[:5]:
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model 1 response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              )
    print("\n数据集回复:")
    print(">>", entry['output'])
    print("\n模型回复:")
    print(">>", entry["model 1 response"])
    print("\n评分:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")

- 注意回复非常冗长；为量化哪个模型更好，我们只需返回分数：

In [ ]:
from tqdm import tqdm


def generate_model_scores(json_data, json_key, client):
    scores = []
    for entry in tqdm(json_data, desc="正在评分"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the number only."
        )
        score = run_chatgpt(prompt, client)
        try:
            scores.append(int(score))
        except ValueError:
            continue

    return scores

- 请注意，回复分数可能会有所不同，因为尽管设置了随机数种子等，OpenAI 的 GPT 模型并非完全确定性的

- 现在对整个数据集应用该评估并计算每个模型的平均分：

In [ ]:
from pathlib import Path

for model in ("model 1 response", "model 2 response"):

    scores = generate_model_scores(json_data, model, client)
    print(f"\n{model}")
    print(f"评分数目: {len(scores)} / {len(json_data)}")
    print(f"平均分: {sum(scores)/len(scores):.2f}\n")

    # 可选：保存分数
    save_path = Path("scores") / f"gpt4-{model.replace(' ', '-')}.json"
    with open(save_path, "w") as file:
        json.dump(scores, file)

- 根据上述评估，我们可以说第 1 个模型明显优于第 2 个模型